In [ ]:


#@title **Install Conda Colab**
#@markdown It will restart the kernel (session), don't worry.
# !pip install -q condacolab
# import condacolab
# condacolab.install()

!pip install -q "https://github.com/conda-incubator/condacolab/archive/main.zip"
import condacolab
condacolab.install(python_version="3.12")

In [ ]:
!pip install -q "https://github.com/conda-incubator/condacolab/archive/main.zip"
import condacolab
condacolab.check()

In [ ]:
!pixi project channel add conda-forge bioconda
!pixi add gromacs
!cat /content/pixi.toml

!gmx --version

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/npt_run', exist_ok=True)
%cd /content/drive/MyDrive/npt_run

In [ ]:
!gmx grompp -f NPT_equilibration_step.mdp -c nvt.gro -t nvt.cpt -p topol.top -o npt.tpr -n index.ndx -maxwarn 1

In [ ]:
#@title **Check if you correctly allocated GPU nodes**

!nvidia-smi

In [ ]:
!pixi run clinfo

In [ ]:
!pixi run gmx mdrun -deffnm npt -nb gpu -pme cpu -ntmpi 1 -ntomp 4

In [10]:
with open('/content/pixi.toml', 'a') as f:
    f.write('\n[system-requirements]\ncuda = "12"\n')

!cat /content/pixi.toml


[workspace]
name = "colab-pixi-kernel"
version = "0.1.0"
channels = ['conda-forge', "bioconda"]
platforms = ["linux-64"]

# These dependencies must be kept more or less in sync with
# https://github.com/googlecolab/backend-info
# Debug from Colab's terminal with:
#  - python3 -m colab_kernel_launcher
#  - python3 -m google.colab._kernel

[dependencies]
python = "3.12.*"
pip = "*"
pillow = "*"
matplotlib-base = "*"
pandas = "2.*"
httplib2 = "*"
google-auth = "2.*"
portpicker = "1.*"
requests = "2.*"
tornado = "6.*"


[constraints]
cuda-version = "12.*"

[pypi-dependencies]
ipython = "==7.*"
ipykernel = "==6.*"
ipyparallel = "==8.*"
jupyter-server = "==2.*"
anywidget = "*"
ipython_genutils = "*"


# Can't be a pypi-dependencies entry because its requirements are way too strict
[tasks]
install-google-colab = "pip install https://github.com/googlecolab/colabtools/archive/refs/heads/main.zip --no-deps"

[system-requirements]
cuda = "12"


In [11]:
!pixi remove gromacs
!pixi add "gromacs=2026.3=*cuda*"

 WARN Encountered 1 warning while parsing the manifest:
  ⚠ the `[system-requirements]` table is deprecated in favor of virtual
  │ packages on `platforms`
    ╭─[/content/pixi.toml:43:1]
 42 │     
 43 │ ╭─▶ [system-requirements]
 44 │ ├─▶ cuda = "12"
    · ╰──── declare these on the `platforms` entries instead
    ╰────
  help: e.g. platforms = [{ name = "my-linux-64", platform = "linux-64", cuda
        = "12" }]

Error:   × dependency `gromacs` was not found in dependencies

 WARN Encountered 1 warning while parsing the manifest:
  ⚠ the `[system-requirements]` table is deprecated in favor of virtual
  │ packages on `platforms`
    ╭─[/content/pixi.toml:43:1]
 42 │     
 43 │ ╭─▶ [system-requirements]
 44 │ ├─▶ cuda = "12"
    · ╰──── declare these on the `platforms` entries instead
    ╰────
  help: e.g. platforms = [{ name = "my-linux-64", platform = "linux-64", cuda
        = "12" }]

✔ Added gromacs=2026.3=*cuda*


In [12]:
!pixi run gmx --version

 WARN Encountered 1 warning while parsing the manifest:
  ⚠ the `[system-requirements]` table is deprecated in favor of virtual
  │ packages on `platforms`
    ╭─[/content/pixi.toml:44:1]
 43 │     
 44 │ ╭─▶ [system-requirements]
 45 │ ├─▶ cuda = "12"
    · ╰──── declare these on the `platforms` entries instead
    ╰────
  help: e.g. platforms = [{ name = "my-linux-64", platform = "linux-64", cuda
        = "12" }]

                   :-) GROMACS - gmx, 2026.3-conda_forge (-:

Executable:   /content/.pixi/envs/default/bin.AVX2_256/gmx
Data prefix:  /content/.pixi/envs/default
Working dir:  /content/drive/MyDrive/npt_run
Command line:
  gmx --version

GROMACS version:     2026.3-conda_forge
Precision:           mixed
Memory model:        64 bit
MPI library:         thread_mpi
MPI version:         built in
OpenMP support:      enabled (GMX_OPENMP_MAX_THREADS = 128)
GPU support:         CUDA
NBNxM GPU setup:     super-cluster 2x2x2 / cluster 8 (cluster-pair splitting on)
SIMD instruction

In [16]:
%%bash --bg
pixi run gmx mdrun -deffnm npt -nb gpu -pme gpu -ntmpi 1 -ntomp 2 -cpt 5 > mdrun_output.log 2>&1

In [18]:
import re, time

def get_latest_step_time(logfile='npt.log'):
    with open(logfile) as f:
        lines = f.readlines()
    last = None
    for i, line in enumerate(lines):
        if line.strip().startswith('Step') and 'Time' in line:
            for j in range(i+1, min(i+3, len(lines))):
                parts = lines[j].split()
                if len(parts) == 2:
                    try:
                        last = (int(parts[0]), float(parts[1]))
                    except ValueError:
                        pass
    return last

step1, t_ps1 = get_latest_step_time()
t_real1 = time.time()
print(f"Campione 1: step {step1}, t = {t_ps1:.2f} ps")

time.sleep(300)  # aspetta 5 minuti

step2, t_ps2 = get_latest_step_time()
t_real2 = time.time()
print(f"Campione 2: step {step2}, t = {t_ps2:.2f} ps")

sim_ns = (t_ps2 - t_ps1) / 1000
real_h = (t_real2 - t_real1) / 3600
ns_per_day = (sim_ns / real_h) * 24 if real_h > 0 else 0

print(f"\nVelocità stimata: {ns_per_day:.2f} ns/day  →  {ns_per_day/24:.3f} ns/h")

Campione 1: step 15000, t = 30.00 ps
Campione 2: step 34000, t = 68.00 ps

Velocità stimata: 10.94 ns/day  →  0.456 ns/h


In [21]:
import subprocess, os

os.chdir('/content/drive/MyDrive/npt_run')

# controlla se mdrun è già in esecuzione in questa sessione
check = subprocess.run(['pgrep', '-f', 'gmx mdrun'], capture_output=True, text=True)
running = bool(check.stdout.strip())

if running:
    print(f"✅ mdrun è già attivo (PID: {check.stdout.strip()}). Nessuna azione necessaria.")
else:
    cpt_exists = os.path.exists('npt.cpt')
    if cpt_exists:
        cmd = 'pixi run gmx mdrun -deffnm npt -cpi npt.cpt -nb gpu -pme gpu -ntmpi 1 -ntomp 2 -cpt 5'
        print("⏯️  Checkpoint trovato: riprendo la simulazione da dove si era interrotta...")
    else:
        cmd = 'pixi run gmx mdrun -deffnm npt -nb gpu -pme gpu -ntmpi 1 -ntomp 2 -cpt 5'
        print("▶️  Nessun checkpoint: avvio una nuova simulazione da zero...")

    with open('mdrun_output.log', 'a') as f:
        subprocess.Popen(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT, start_new_session=True)
    print(f"Lanciato in background: {cmd}")

✅ mdrun è già attivo (PID: 10143
10170
10204
10231). Nessuna azione necessaria.
